In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

customers = spark.createDataFrame([
    (1,"John ","Hyderabad"),
    (2,"Alice","Chennai"),
    (3,None,"Bangalore")
],["customer_id","name","city"])

In [0]:
cars = spark.createDataFrame([
    (101,"Toyota","Camry",30000),
    (102,"Honda","Civic",20000),
    (103,"Hyundai","i20",15000)
],["car_id","brand","model","price"])

In [0]:

sales = spark.createDataFrame([
    (1,1,101,"2024-01-01",1),
    (2,2,102,"2024-01-02",2),
    (3,99,103,"2024-01-03",1)
],["sale_id","customer_id","car_id","sale_date","quantity"])

In [0]:
from pyspark.sql.functions import to_date, col

sales = sales.withColumn("sale_date", to_date(col("sale_date")))

df = sales.join(customers, "customer_id").join(cars, "car_id")

df.groupBy("customer_id").sum("price").show()

+-----------+----------+
|customer_id|sum(price)|
+-----------+----------+
|          1|     30000|
|          2|     20000|
+-----------+----------+



In [0]:
customers_clean= customers.withColumn("name", trim(col("name"))).fillna({"name":"unknown"}).show()

+-----------+-------+---------+
|customer_id|   name|     city|
+-----------+-------+---------+
|          1|   John|Hyderabad|
|          2|  Alice|  Chennai|
|          3|unknown|Bangalore|
+-----------+-------+---------+



In [0]:
cars_clean=cars.withColumn("price", abs(col("price"))).show()

+------+-------+-----+-----+
|car_id|  brand|model|price|
+------+-------+-----+-----+
|   101| Toyota|Camry|30000|
|   102|  Honda|Civic|20000|
|   103|Hyundai|  i20|15000|
+------+-------+-----+-----+



In [0]:
sales_clean=sales.join(customers ,"customer_id","inner").show()

+-----------+-------+------+----------+--------+-----+---------+
|customer_id|sale_id|car_id| sale_date|quantity| name|     city|
+-----------+-------+------+----------+--------+-----+---------+
|          1|      1|   101|2024-01-01|       1|John |Hyderabad|
|          2|      2|   102|2024-01-02|       2|Alice|  Chennai|
+-----------+-------+------+----------+--------+-----+---------+



In [0]:
customers_clean = customers.withColumn("name", trim(col("name"))).fillna({"name":"unknown"})
invalid_sales = sales.join(customers_clean, "customer_id", "left_anti")
invalid_sales.show()

+-----------+-------+------+----------+--------+
|customer_id|sale_id|car_id| sale_date|quantity|
+-----------+-------+------+----------+--------+
|         99|      3|   103|2024-01-03|       1|
+-----------+-------+------+----------+--------+



In [0]:
sales_clean = sales.join(customers, "customer_id", "inner")
cars_clean = cars.withColumn("price", abs(col("price")))

revenue_per_customer = (sales_clean.join(cars_clean, "car_id")
    .groupBy("customer_id").sum("price")
    .withColumnRenamed("sum(price)", "total_revenue"))
revenue_per_customer.show()

+-----------+-------------+
|customer_id|total_revenue|
+-----------+-------------+
|          1|        30000|
|          2|        20000|
+-----------+-------------+



In [0]:
cars_clean=cars.withColumn("price",abs(col("price")))
cars_per_brand=cars_clean.groupBy("brand").count()
cars_per_brand.show()

+-------+-----+
|  brand|count|
+-------+-----+
| Toyota|    1|
|  Honda|    1|
|Hyundai|    1|
+-------+-----+



In [0]:
customers_clean = customers.withColumn("name", trim(col("name"))).fillna({"name":"unknown"})
customer_revenue = df.join(customers_clean.drop("name", "city"), "customer_id").groupBy("city").sum("price").withColumnRenamed("sum(price)", "city_revenue")
customer_revenue.show()

+---------+------------+
|     city|city_revenue|
+---------+------------+
|Hyderabad|       30000|
|  Chennai|       20000|
+---------+------------+



In [0]:
sales_clean.createOrReplaceTempView("sales_data")
customers_clean.createOrReplaceTempView("customers")
cars_clean.createOrReplaceTempView("cars")

In [0]:
%sql
SELECT city, customer_id, total_revenue
FROM (
    SELECT c.city, s.customer_id,
           SUM(ca.price) AS total_revenue,
           ROW_NUMBER() OVER (PARTITION BY c.city ORDER BY SUM(ca.price) DESC) rn
    FROM sales_data s
    JOIN customers c ON s.customer_id = c.customer_id
    JOIN cars ca ON s.car_id = ca.car_id
    GROUP BY c.city, s.customer_id
) t
WHERE rn <= 3;

city,customer_id,total_revenue
Chennai,2,20000
Hyderabad,1,30000


In [0]:
%sql
select customer_id, count(sale_id) as purchases
from sales_data 
group by(customer_id)
having count(*)>1;

customer_id,purchases


In [0]:
%sql
select month(s.sale_date) as month, sum(c.price) as revenue
from sales_data s
join cars c on s.car_id = c.car_id
group by month(s.sale_date)

month,revenue
1,50000


In [0]:

customer_revenue.write.mode("overwrite").saveAsTable("car_sales_output")